# 01. Bayes Filter와 Recursive State Estimation

Probabilistic Robotics의 출발점은 로봇 상태를 하나의 값이 아니라 **belief distribution**으로 표현하는 것이다.

$$bel(x_t)=p(x_t\mid z_{1:t},u_{1:t})$$

Bayes filter는 두 단계로 반복된다.

1. Prediction: motion model로 belief를 밀어낸다.
2. Correction: sensor model로 관측과 맞는 상태를 더 믿는다.

$$\bar{bel}(x_t)=\int p(x_t\mid u_t,x_{t-1})bel(x_{t-1})dx_{t-1}$$
$$bel(x_t)=\eta p(z_t\mid x_t)\bar{bel}(x_t)$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
os.makedirs('assets', exist_ok=True)

for font_name in ['Nanum Gothic', 'AppleGothic', 'Malgun Gothic']:
    if any(font.name == font_name for font in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False

## 1. 1D 복도 로봇 예제

로봇은 원형 복도 위에 있고, 명령 `move +1`은 대부분 한 칸 이동하지만 가끔 미끄러진다.
관측은 현재 칸이 문인지 아닌지 알려주지만 노이즈가 있다.

In [ ]:
world = np.array([0, 1, 0, 0, 1, 0, 1, 0, 0, 1])  # 1 = door
n = len(world)
bel = np.ones(n) / n

motion_kernel = {-1: 0.1, 0: 0.15, 1: 0.75}
p_hit = 0.85
p_false = 0.15

def predict(bel, u=1):
    out = np.zeros_like(bel)
    for i in range(n):
        for slip, prob in motion_kernel.items():
            j = (i + u + slip) % n
            out[j] += prob * bel[i]
    return out

def correct(bel, z_door):
    likelihood = np.where(world == z_door, p_hit, p_false)
    out = likelihood * bel
    return out / out.sum()

observations = [1, 0, 1, 1, 0]
history = [bel.copy()]
for z in observations:
    bel = predict(bel, u=1)
    history.append(bel.copy())
    bel = correct(bel, z)
    history.append(bel.copy())
history = np.array(history)

fig, axes = plt.subplots(len(history), 1, figsize=(10, 10), sharex=True)
for k, ax in enumerate(axes):
    ax.bar(np.arange(n), history[k], color='#534AB7')
    ax.set_ylim(0, history.max()*1.15)
    ax.set_ylabel(f't{k}')
    ax.grid(axis='y', alpha=0.2)
axes[-1].set_xlabel('cell index')
plt.suptitle('Bayes filter: prediction과 correction 반복')
plt.tight_layout()
plt.savefig('assets/01_bayes_filter_1d.png', dpi=150, bbox_inches='tight')
plt.show()

print('final belief=', np.round(bel, 3))
print('most likely cell=', int(np.argmax(bel)))

## 2. Markov Assumption

Bayes filter는 현재 상태 $x_t$가 직전 상태와 현재 입력에만 의존한다고 가정한다.

$$p(x_t\mid x_{0:t-1},u_{1:t}) = p(x_t\mid x_{t-1},u_t)$$

이 가정 덕분에 모든 과거 데이터를 직접 들고 있지 않고 belief만 업데이트하면 된다.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
labels = ['x(t-1)', 'u(t)', 'x(t)', 'z(t)', 'bel(t)']
pos = np.array([[0,0], [1.5,0.7], [3,0], [4.5,0.7], [6,0]])
for (x,y), lab in zip(pos, labels):
    ax.text(x, y, lab, ha='center', va='center', fontsize=12,
            bbox=dict(boxstyle='round,pad=0.35', fc='white', ec='#534AB7', lw=1.8))
for a,b in [(0,2), (1,2), (2,3), (2,4), (3,4)]:
    ax.annotate('', xy=pos[b], xytext=pos[a], arrowprops=dict(arrowstyle='->', lw=1.8, color='gray'))
ax.axis('off')
ax.set_title('Recursive state estimation dependency')
plt.savefig('assets/01_markov_assumption.png', dpi=150, bbox_inches='tight')
plt.show()

## 요약

| 개념 | 의미 | 책 커리큘럼 연결 |
|------|------|------------------|
| Belief | 상태 확률분포 | Ch.2 Recursive State Estimation |
| Motion model | $p(x_t\mid u_t,x_{t-1})$ | Ch.5 Robot Motion |
| Sensor model | $p(z_t\mid x_t)$ | Ch.6 Robot Perception |
| Bayes filter | 예측/보정 반복 | Gaussian, histogram, particle filter의 공통 뼈대 |